# RAG Benchmark — Google Colab Runner

**Stack**
| Component | Choice |
|-----------|--------|
| Generator LLM | `llama-3.1-8b-instant` via Groq (free, fast) |
| Judge LLM | `llama-3.3-70b-versatile` via Groq (free, stronger) |
| Embeddings | `BAAI/bge-large-en-v1.5` — runs on T4 GPU |
| Vector store | FAISS (exact search) |
| Evaluation | RAGAS (faithfulness + answer relevancy) |

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Open the 🔑 Secrets panel (left sidebar) and add:
   - `GROQ_API_KEY` — free key from [console.groq.com](https://console.groq.com)
   - `GITHUB_TOKEN` — a GitHub personal access token with `repo` scope (for pushing results)
3. Run cells top to bottom.

## 1 · Clone repository

In [ ]:
# ── Change this to your repo ──────────────────────────────────────────────────
GITHUB_USER = "your-username"          # e.g. "shreya"
REPO_NAME   = "rag-benchmark"          # e.g. "rag-benchmark"
BRANCH      = "main"
# ─────────────────────────────────────────────────────────────────────────────

import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Clone (or pull if already cloned from a previous session)
if not os.path.exists(f"/content/{REPO_NAME}/.git"):
    !git clone --branch {BRANCH} {REPO_URL} /content/{REPO_NAME}
else:
    !git -C /content/{REPO_NAME} pull

%cd /content/{REPO_NAME}
print(f"Working directory: {os.getcwd()}")

## 2 · Install dependencies

In [ ]:
%%capture install_output
# Colab already has torch + numpy; install project-specific packages
!pip install -q \
    groq \
    langchain-groq \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    rank-bm25 \
    spacy \
    networkx \
    ragas \
    datasets \
    wikipedia-api \
    arxiv \
    psutil \
    python-dotenv \
    tiktoken

!python -m spacy download -q en_core_web_sm

print("Dependencies installed.")

## 3 · Configure secrets

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Sanity check — never print the actual key
key = os.environ.get("GROQ_API_KEY", "")
print(f"GROQ_API_KEY set: {'yes (' + key[:8] + '...)' if key else 'NO — check Secrets'}")

## 4 · Verify environment

In [ ]:
import torch, psutil

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU only)"
ram_gb = psutil.virtual_memory().total / 1e9

print(f"GPU  : {gpu}")
print(f"RAM  : {ram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

# Confirm bge-large will use GPU
print(f"\nEmbeddings will use: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

## 5 · Run benchmark

Choose dataset: `small` (50 Wikipedia articles) · `medium` (500 arXiv papers) · `large` (2400 Kubernetes docs)

In [ ]:
DATASET = "small"   # change to "medium" or "large" as needed

!python run_benchmark.py --dataset {DATASET}

## 6 · Display results

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Anchor to repo root — works in both Colab (/content/rag-benchmark) and VS Code
results_dir = Path.cwd() / "results" / "benchmark_tables"

# Only look for benchmark run files — exclude QA cache files (qa_*.json)
run_files    = sorted(results_dir.glob("run_*.json"))
summary_file = results_dir / "summary.json"

if not run_files and not summary_file.exists():
    print("No benchmark results yet — run Cell 5 first, then come back here.")
else:
    rows = []
    # Prefer summary.json (written after all runs complete); fall back to per-run files
    if summary_file.exists():
        with open(summary_file) as f:
            data = json.load(f)
        rows = data if isinstance(data, list) else [data]
    else:
        for f in run_files:
            with open(f) as fh:
                rows.append(json.load(fh))

    records = []
    for r in rows:
        records.append({
            "Run ID":          r.get("run_id", "-"),
            "Architecture":    r.get("architecture", "-"),
            "Dataset":         r.get("dataset", "-"),
            "Recall@5":        round(r.get("retrieval_metrics", {}).get("recall_at_5",     0), 3),
            "MRR":             round(r.get("retrieval_metrics", {}).get("mrr",             0), 3),
            "P50 latency (s)": round(r.get("system_metrics",   {}).get("p50_latency",      0), 3),
            "Faithfulness":    round(r.get("quality_metrics",  {}).get("faithfulness",     0), 3),
            "Ans. relevancy":  round(r.get("quality_metrics",  {}).get("answer_relevancy", 0), 3),
            "$/query":         round(r.get("cost",             {}).get("per_query",        0), 5),
        })

    df = pd.DataFrame(records).sort_values(["Dataset", "Architecture"])
    print(f"Showing {len(df)} run(s) from {results_dir}\n")
    display(df)

## 7 · Push results to GitHub

In [ ]:
import subprocess
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

# Configure git identity for this session
!git config user.email "colab@rag-benchmark"
!git config user.name  "Colab Runner"

# Stage results and any updated cache files
!git add results/ datasets/processed/ --force
!git status --short

result = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    capture_output=True
)

if result.returncode == 0:
    print("Nothing new to commit — results already up to date.")
else:
    import datetime
    timestamp = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    !git commit -m "chore: benchmark results {timestamp}"
    !git push
    print("Results pushed to GitHub.")